
<center> <a href="https://github.com/CyConProject?tab=repositories">
  <img src="https://github.com/CyConProject/Lab/blob/main/Figures/CyCon.png?raw=true" alt="logo" width="80" >
</a>
 </center>

## Regression Projects with Keras

Concrete mix design is a classic **civil-engineering regression problem**: given the ingredients and curing age of a batch, can a model estimate its **compressive strength**? In this lab, a deep learning workflow in **Keras** is used to predict strength from mixture variables.

This lab builds on the ideas from **Module 3-2 (frameworks)** and **Module 3-3 (implementing regression in Keras)**. The focus is on the full workflow: loading data, preparing predictors and targets, building a network, training it, and using it to support engineering decisions.

### Objectives

1. Explain why **Keras** is a strong entry point for deep learning projects.
2. Load and inspect a civil-engineering regression dataset.
3. Split the dataset into **predictors** and **target**, then normalize the inputs.
4. Build a **Sequential** neural network for regression in Keras.
5. Train the model and evaluate its predictive performance.
6. Use `predict()` to estimate the strength of new candidate concrete mixes.



## 1. Recap: Why Keras for Regression?

When engineers begin using deep learning, they usually encounter three names very quickly: **TensorFlow**, **PyTorch**, and **Keras**.

- **TensorFlow** is widely used in production systems and has a large ecosystem.
- **PyTorch** is especially popular in research and custom model development.
- **Keras** provides a simpler, higher-level interface that makes model building much easier to read and write.

For this lab, **Keras** is a strong fit because it allows a useful neural network to be defined in only a few lines of code. That simplicity makes it easier to focus on the engineering problem rather than framework complexity.

### Regression vs. Classification

In **regression**, the model predicts a continuous numerical value, such as:

- concrete compressive strength,
- settlement magnitude,
- pavement roughness,
- or traffic demand.

In this notebook, the network outputs **one number**: the predicted concrete strength in MPa.

### Workflow for this lab

1. Load a concrete-mixture dataset.
2. Separate **input features** from the **strength target**.
3. Normalize the feature columns.
4. Build a dense neural network with **ReLU** hidden layers.
5. Train the model using **Adam** and **mean squared error**.
6. Evaluate predictions and test new candidate mixes.



## 2. Part I – Setup and Data Loading

This lab uses the `Concrete_Data.csv` dataset hosted in the course GitHub repository.

#### `Concrete_Data.csv`:

The regression problem focuses on predicting the concrete compressive strength using its components as well as its age. The variables are listed in the same order as the numerical values in the database rows.

| Name | Data Type | Measurement | Description |
|---|---|---|---|
| **Cement** | quantitative | kg in a m3 mixture | Input Variable |
| **Blast Furnace Slag** | quantitative | kg in a m3 mixture | Input Variable |
| **Fly Ash** | quantitative | kg in a m3 mixture | Input Variable |
| **Water** | quantitative | kg in a m3 mixture | Input Variable |
| **Superplasticizer** | quantitative | kg in a m3 mixture | Input Variable |
| **Coarse Aggregate** | quantitative | kg in a m3 mixture | Input Variable |
| **Fine Aggregate** | quantitative | kg in a m3 mixture | Input Variable |
| **Age** | quantitative | Day (1~365) | Input Variable |
| **Concrete compressive strength** | quantitative | MPa | Output Variable |

**Data source:** [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength)

For programming convenience, the notebook renames the original column labels into shorter Python-friendly names such as `cement`, `slag`, `flyash`, `age_days`, and `strength_mpa`.


In [ ]:

# Uncomment this line if packages need to be installed locally.
# !pip install tensorflow pandas numpy scikit-learn matplotlib

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

np.random.seed(42)
tf.random.set_seed(42)


In [ ]:

DATA_URL = "https://raw.githubusercontent.com/CyConProject/Lab/main/Datasets/Concrete_Data.csv"


def load_concrete_strength_data(url=DATA_URL):
    rename_map = {
        "cement": "cement",
        "slag": "blast_furnace_slag",
        "flyash": "fly_ash",
        "water": "water",
        "superplasticizer": "superplasticizer",
        "coarseaggregate": "coarse_aggregate",
        "fineaggregate": "fine_aggregate",
        "age": "age_days",
        "csMPa": "strength_mpa",
    }

    df = pd.read_csv(url)
    df = df.rename(columns=rename_map)
    return df


concrete_data = load_concrete_strength_data()
concrete_data.head()



### 2.1 Inspect the dataset

Before training a network, it is good engineering practice to understand the data first.

Look for:

- the number of rows and columns,
- missing values,
- rough ranges of each feature,
- and whether the target appears reasonable.

> **Exercise 1:** Report the dataset shape, list missing values by column, and show descriptive statistics.


In [ ]:
# To Do



<details>
<summary>Solutions (click to expand)</summary>

```python
print("Shape:", concrete_data.shape)
print("Missing values by column:")
print(concrete_data.isnull().sum())

print("Summary statistics:")
print(concrete_data.describe().T)
```

</details>


In [ ]:
summary_table = concrete_data.describe().T
summary_table



## 3. Part II – Predictors, Target, and Normalization

For supervised learning, the table is split into:

- **predictors**: the input features used by the model,
- **target**: the quantity to predict.

Here, the target is `strength_mpa`, and the predictors are all remaining columns.

After that, the dataset is divided into **training** and **test** subsets. The training data teaches the network; the test data provides an honest performance check.

Finally, the predictor columns are normalized using the **training-set mean and standard deviation** only. This avoids leaking information from the test set.

> **Exercise 2:** Create `X` and `y`, split the dataset into training and test sets, and standardize the predictors.


In [ ]:

# To Do
# X = ...
# y = ...
# X_train, X_test, y_train, y_test = ...
# train_mean = ...
# train_std = ...
# X_train_scaled = ...
# X_test_scaled = ...



<details>
<summary>Solutions (click to expand)</summary>

```python
X = concrete_data.drop(columns="strength_mpa")
y = concrete_data["strength_mpa"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

train_mean = X_train.mean()
train_std = X_train.std().replace(0, 1)

X_train_scaled = (X_train - train_mean) / train_std
X_test_scaled = (X_test - train_mean) / train_std
```

</details>


In [ ]:
print("Number of predictors:", X_train_scaled.shape[1])
print("Training samples:", X_train_scaled.shape[0])
print("Test samples:", X_test_scaled.shape[0])



## 4. Part III – Build a Regression Network in Keras

In Keras, a **Sequential** model is a convenient choice when the network is a simple stack of layers.

For this lab, the model will use:

- an **input layer** that matches the number of predictor columns,
- two hidden **Dense** layers with **5 neurons each** and **ReLU** activation,
- and one output node for the predicted strength.

Because this is a regression task:

- the output layer has **one node**,
- and it uses a **linear output** (no extra activation is needed).

When the model is compiled, it uses:

- **Adam** as the optimizer,
- **MSE** as the loss,
- and **MAE** as an additional metric for easier interpretation.

> **Exercise 3:** Complete the function below so it builds a regression model with two hidden layers of 5 neurons each.


In [ ]:

def build_regression_model(n_features):
    model = Sequential()

    # To Do
    # model.add(...)
    # model.add(...)
    # model.add(...)
    # model.add(...)

    # model.compile(...)
    return model



<details>
<summary>Solutions (click to expand)</summary>

```python
def build_regression_model(n_features):
    model = Sequential()
    model.add(Input(shape=(n_features,)))
    model.add(Dense(5, activation="relu"))
    model.add(Dense(5, activation="relu"))
    model.add(Dense(1))

    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model
```

</details>


In [ ]:
baseline_model = build_regression_model(X_train_scaled.shape[1])
baseline_model.summary()



## 5. Part IV – Train and Evaluate the Model

The model is now ready for training.

A few notes:

- `fit()` performs the training loop.
- `validation_split=0.2` keeps part of the training data aside for validation during training.
- `epochs` controls how many full passes are made through the training data.

After training, the model is evaluated on the test set and a **predicted vs. measured** plot is inspected.


In [ ]:

baseline_model = build_regression_model(X_train_scaled.shape[1])

history = baseline_model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.20,
    epochs=100,
    batch_size=32,
    verbose=0,
)

history_df = pd.DataFrame(history.history)
history_df.tail()


In [ ]:

plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("MSE During Training")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.show()


In [ ]:

plt.figure(figsize=(7, 4))
plt.plot(history.history["mae"], label="Training MAE")
plt.plot(history.history["val_mae"], label="Validation MAE")
plt.title("MAE During Training")
plt.xlabel("Epoch")
plt.ylabel("MAE (MPa)")
plt.legend()
plt.show()


In [ ]:

test_loss, test_mae = baseline_model.evaluate(X_test_scaled, y_test, verbose=0)
y_test_pred = baseline_model.predict(X_test_scaled, verbose=0).flatten()

rmse = mean_squared_error(y_test, y_test_pred) ** 0.5
r2 = r2_score(y_test, y_test_pred)

print(f"Test MSE:  {test_loss:.3f}")
print(f"Test MAE:  {test_mae:.3f} MPa")
print(f"Test RMSE: {rmse:.3f} MPa")
print(f"Test R^2:  {r2:.3f}")


In [ ]:

plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_test_pred, alpha=0.7)

line_min = min(y_test.min(), y_test_pred.min())
line_max = max(y_test.max(), y_test_pred.max())
plt.plot([line_min, line_max], [line_min, line_max], "k--")

plt.xlabel("Measured strength (MPa)")
plt.ylabel("Predicted strength (MPa)")
plt.title("Measured vs. Predicted Concrete Strength")
plt.show()



### 5.1 Engineering interpretation

A perfect model would place every point exactly on the dashed 45-degree line. In practice, some scatter is expected.

- Lower **MAE** means the average prediction error is smaller.
- Lower **RMSE** penalizes large mistakes more strongly.
- Higher **R^2** means the model explains more of the variation in strength.

For engineering use, this model should support judgment rather than replace it. Material variability, curing conditions, batching quality, and site practices can all affect actual field performance.



## 6. Part V – Predict New Mix Designs

One of the most useful moments in a regression project is when the model begins answering practical design questions.

Suppose several candidate concrete mixes are being compared for a civil works project. Which mix is expected to achieve the highest compressive strength?

> **Exercise 4:** Use the trained model to predict the strength of the three candidate mixes below, then identify the strongest option.


In [ ]:

candidate_mixes = pd.DataFrame([
    {
        "cement": 350,
        "blast_furnace_slag": 80,
        "fly_ash": 40,
        "water": 180,
        "superplasticizer": 8,
        "coarse_aggregate": 1040,
        "fine_aggregate": 760,
        "age_days": 28,
    },
    {
        "cement": 420,
        "blast_furnace_slag": 20,
        "fly_ash": 0,
        "water": 165,
        "superplasticizer": 10,
        "coarse_aggregate": 1025,
        "fine_aggregate": 745,
        "age_days": 28,
    },
    {
        "cement": 300,
        "blast_furnace_slag": 120,
        "fly_ash": 60,
        "water": 155,
        "superplasticizer": 12,
        "coarse_aggregate": 1005,
        "fine_aggregate": 780,
        "age_days": 56,
    },
], index=["Mix A", "Mix B", "Mix C"])

candidate_mixes


In [ ]:

# To Do
# candidate_scaled = ...
# candidate_predictions = ...
# results = ...



<details>
<summary>Solutions (click to expand)</summary>

```python
candidate_scaled = (candidate_mixes - train_mean) / train_std
candidate_predictions = baseline_model.predict(candidate_scaled, verbose=0).flatten()

results = candidate_mixes.copy()
results["predicted_strength_mpa"] = candidate_predictions
results = results.sort_values("predicted_strength_mpa", ascending=False)
print(results[["predicted_strength_mpa"]])
```

</details>



## 7. Part VI – Mini Project: Compare Two Network Designs

A useful engineering question is whether a more complex model actually improves performance.

Below, two models are compared:

- a **baseline** model with hidden layers `[5, 5]`,
- and a **wider** model with hidden layers `[10, 10]`.

> **Experiment:** Train both models for the same number of epochs and compare their test metrics. Does the wider network improve the results enough to justify the extra complexity?


In [ ]:

def build_wider_model(n_features):
    model = Sequential([
        Input(shape=(n_features,)),
        Dense(10, activation="relu"),
        Dense(10, activation="relu"),
        Dense(1),
    ])
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model


def evaluate_model(model_builder, name, X_train_scaled, y_train, X_test_scaled, y_test, epochs=100):
    model = model_builder(X_train_scaled.shape[1])
    model.fit(
        X_train_scaled,
        y_train,
        validation_split=0.20,
        epochs=epochs,
        batch_size=32,
        verbose=0,
    )

    test_mse, test_mae = model.evaluate(X_test_scaled, y_test, verbose=0)
    y_pred = model.predict(X_test_scaled, verbose=0).flatten()
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)

    return {
        "model": name,
        "test_mse": test_mse,
        "test_mae": test_mae,
        "test_rmse": rmse,
        "test_r2": r2,
    }

comparison = pd.DataFrame([
    evaluate_model(build_regression_model, "Baseline [5, 5]", X_train_scaled, y_train, X_test_scaled, y_test),
    evaluate_model(build_wider_model, "Wider [10, 10]", X_train_scaled, y_train, X_test_scaled, y_test),
])

comparison



## 8. Reflection and Extensions

1. Try changing the hidden-layer sizes. Does a wider network help?
2. Train for fewer or more epochs. When do diminishing returns begin?
3. Remove one predictor, such as `age_days`, and observe how much performance changes.
4. Compare training with normalized inputs versus raw inputs.
5. Replace the concrete-strength problem with another civil-engineering regression task, such as settlement prediction, traffic-flow estimation, or asphalt stiffness.

> **Key idea:** Keras makes deep learning accessible because it allows the workflow to move from data preparation to model training and prediction with very little code. That simplicity is especially valuable when the main goal is solving an engineering problem.
